# Dog Face Detection

REF

[Object Detection](https://medium.com/@RobuRishabh/understanding-and-implementing-faster-r-cnn-248f7b25ff96)

In [8]:
import torch 
import torchvision
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import torch.utils.data
from PIL import Image
import os
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

In [6]:
model = fasterrcnn_resnet50_fpn(pretrained=True)

# get number of input features for the classifier
in_features = model.roi_heads.box_predictor.cls_score.in_features
# replace the pre-trained head with a new one
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)

/home/eclipse/anaconda3/envs/DOG_EMOTION/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/eclipse/anaconda3/envs/DOG_EMOTION/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /home/eclipse/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:01<00:00, 113MB/s]  


In [41]:
def get_transform(train):
    transforms = [T.ToTensor()]
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    return T.Compose(transforms)


class DogDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None,):
        self.root = root
        self.imgs = []
        images_dir = os.path.join(root, "Images")
        for subdir, _, files in os.walk(images_dir):
            for file in files:
                if file.lower().endswith(('jpg', 'jpeg', 'png')):
                    self.imgs.append(os.path.join(subdir, file))
                    
        self.annot_root = os.path.join(root, "Annotations")

        self.transforms = transforms
        
    def parse_xml(self, xml_file):
        """Parse XML annotation file to extract bounding box information."""
        tree = ET.parse(xml_file)
        root = tree.getroot()
        boxes = []
        
        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)
            boxes.append([xmin, ymin, xmax, ymax])
        
        return boxes if boxes else None
    
    def __len__(self):
        return len(self.imgs)
        
    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        img = Image.open(img_path).convert("RGB")
        
        rel_path = os.path.relpath(img_path, os.path.join(self.root, "Images"))
        annot_path = os.path.join(self.annot_root, os.path.splitext(rel_path)[0])

        boxes = self.parse_xml(annot_path)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = torch.ones((boxes.shape[0],), dtype=torch.int64)
        target["image_id"] = torch.tensor([idx])

        if self.transforms:
            img = self.transforms(img)
        
        return img, target

In [42]:
dataset = DogDataset(root='../standford_dog_data', transforms=get_transform(train=True))
dataset_test = DogDataset(root='../standford_dog_data', transforms=get_transform(train=False))

indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-50])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-50:])



In [43]:
data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=216, shuffle=True, num_workers=4,
        collate_fn=lambda x: tuple(zip(*x))
    )
data_loader_test = torch.utils.data.DataLoader(
        dataset_test, batch_size=1, shuffle=False, num_workers=4,
        collate_fn=lambda x: tuple(zip(*x))
    )


In [44]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# Set up the optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, 
                                                   weight_decay=0.0005)
# Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, 
                                                               gamma=0.1)
# Train the model
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

   # Training loop
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass
        losses.backward()
        optimizer.step()
        train_loss += losses.item()

    # Update the learning rate
    lr_scheduler.step()
    print(f'Epoch: {epoch + 1}, Loss: {train_loss / len(data_loader)}')
print("Training complete!")


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/eclipse/anaconda3/envs/DOG_EMOTION/lib/python3.9/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/eclipse/anaconda3/envs/DOG_EMOTION/lib/python3.9/site-packages/torch/utils/data/_utils/fetch.py", line 50, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
  File "/home/eclipse/anaconda3/envs/DOG_EMOTION/lib/python3.9/site-packages/torch/utils/data/dataset.py", line 420, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/home/eclipse/anaconda3/envs/DOG_EMOTION/lib/python3.9/site-packages/torch/utils/data/dataset.py", line 420, in <listcomp>
    return [self.dataset[self.indices[idx]] for idx in indices]
  File "/tmp/ipykernel_7989/3325704599.py", line 48, in __getitem__
    boxes = self.parse